# Posterior distortion, statistical semantics and the matched native comparison
[Proof](../12_posterior_precision_distortion.md). Retain the original Gaussian positive controls; finite witnesses are not trained-model evidence.

In [ ]:
import numpy as np
from experiments.sampled_conditioning.core import gaussian_kl, precision_certificate
from experiments.sampled_conditioning.parameterization_controls import bounds, denoiser_gap
rng=np.random.default_rng(12)
errors=[]; rows=[]
for i in range(100):
    a,b=rng.normal(size=(4,4)),rng.normal(size=(4,4))
    p,q=a@a.T+np.eye(4),b@b.T+np.eye(4)
    eta,etq=rng.normal(size=4),rng.normal(size=4)
    result=precision_certificate(p,eta,q,etq)
    direct=gaussian_kl(np.linalg.solve(p,eta),np.linalg.inv(p),np.linalg.solve(q,etq),np.linalg.inv(q))
    tight=bounds(p,eta,q,etq)
    errors.append(abs(direct-result['kl_nats']))
    assert direct<=tight['tight_bound']+1e-8<=tight['old_bound']+2e-8
    np.testing.assert_allclose(result['bound_nats'],tight['old_bound'],rtol=1e-8,atol=1e-8)
    rows.append([i,direct,tight['old_bound'],tight['tight_bound'],tight['tight_bound']/direct])
assert max(errors)<1e-9
rows=np.array(rows)
print('max independent KL discrepancy',max(errors))
print('columns: example, exact KL, old bound, tight bound, tight/exact')
print(rows[:8])
print('tight/exact min, median, max',np.min(rows[:,4]),np.median(rows[:,4]),np.max(rows[:,4]))
print('minimum exact and minimum bound example',int(rows[np.argmin(rows[:,1]),0]),int(rows[np.argmin(rows[:,3]),0]))

## Joint natural-parameter error and a zero perturbation
Preserve interaction, not just separate error norms.

In [ ]:
p=np.eye(2);q=np.array([[1.,.4],[.4,1.]])
values=[precision_certificate(p,np.array([s,0.]),q,np.array([s,0.]))['kl_nats'] for s in [0.,1.,10.]]
assert values[2]>values[1]>values[0]
zero=bounds(p,np.ones(2),p,np.ones(2))
assert zero['exact_kl']==zero['tight_bound']==0
h=np.array([1.,2.]);e=q-p
cancel=bounds(p,h,q,h+e@h);amplify=bounds(p,h,q,h-e@h)
assert cancel['mean_residual_squared']<1e-25
assert amplify['exact_kl']>cancel['exact_kl']
print('scale test',values,'cancel',cancel,'amplify',amplify)

## Target-space denoiser and ranking control
Both predictors use the same noisy target from the true law.

In [ ]:
mu=np.array([.2,-.3]);mq=np.array([-.1,.4])
v=np.array([[.8,.2],[.2,.5]]);w=np.diag([.6,1.2]);a,s=.8,.6
m=a*a*v+s*s*np.eye(2);n=a*a*w+s*s*np.eye(2)
z=np.random.default_rng(2).multivariate_normal(a*mu,m,size=100000)
f=s*np.linalg.solve(m,(z-a*mu).T).T
fq=s*np.linalg.solve(n,(z-a*mq).T).T
losses=np.sum((f-fq)**2,axis=1);expected=denoiser_gap(mu,v,mq,w,a,s)
se=losses.std(ddof=1)/np.sqrt(len(losses))
assert abs(losses.mean()-expected)<5*se
print('Monte Carlo, exact discrepancy, SE',losses.mean(),expected,se)
assert denoiser_gap(mu,v,mq,w,0.,1.)==0
p=np.zeros(2);a=np.array([0.,.9]);b=np.array([1.,0.]);eye=np.eye(2);L=np.array([[0.,1.]])
assert gaussian_kl(p,eye,a,eye)<gaussian_kl(p,eye,b,eye)
assert gaussian_kl(L@p,L@eye@L.T,L@a,L@eye@L.T)>gaussian_kl(L@p,L@eye@L.T,L@b,L@eye@L.T)

## Gaussian score and a public covariance floor
A proper objective for the first two moments is not a full-distribution representation. A finite covariance constraint changes its optimum.

In [ ]:
mu=np.array([.2,-.3]); V=np.array([[.8,.2],[.2,.5]])
m=np.array([-.1,.4]); S=np.array([[.6,.1],[.1,1.2]])
def expected_score(mu,V,m,S):
    delta=mu-m
    return .5*(np.linalg.slogdet(S)[1]+np.trace(np.linalg.solve(S,V))+delta@np.linalg.solve(S,delta))
excess=expected_score(mu,V,m,S)-expected_score(mu,V,mu,V)
np.testing.assert_allclose(excess,gaussian_kl(mu,V,m,S),atol=1e-12)
collapse=[.5*2*np.log(e) for e in [1.,1e-4,1e-8]]
assert collapse[2]<collapse[1]<collapse[0]
Vfloor=np.diag([.01,2.]); optimum=np.diag([.1,2.])
for diagonal in ([.1,.1],[.2,2.],[.1,3.]):
    assert expected_score(np.zeros(2),Vfloor,np.zeros(2),np.diag(diagonal))>=expected_score(np.zeros(2),Vfloor,np.zeros(2),optimum)-1e-12
print('Gaussian score excess / KL:',excess)
print('interpolated-mean logdet collapse:',collapse)
print('closed constrained optimum eigenvalues:',np.diag(optimum))

## Same-moment distinct smooth distributions
Integrate the actual compressed Bayes gap, then refine the grid. An ordinary code retaining law identity resolves this example equally well as a mixed code.

In [ ]:
def shape_gap(points):
    z=np.linspace(-10,10,points); alpha,sigma=.8,.6
    means=[np.array([0.]),np.array([-.95,.95])]
    variances=[np.array([1.]),np.array([.0975,.0975])]
    ps=[];fs=[]
    for mean,var in zip(means,variances):
        obsvar=alpha*alpha*var+sigma*sigma
        c=np.exp(-.5*(z[:,None]-alpha*mean)**2/obsvar)/np.sqrt(2*np.pi*obsvar)
        density=c.mean(1)
        derivative=(-(z[:,None]-alpha*mean)/obsvar*c).mean(1)
        ps.append(density);fs.append(-sigma*derivative/density)
    integrand=.5*ps[0]*ps[1]/(ps[0]+ps[1])*(fs[0]-fs[1])**2
    return float(np.sum((integrand[1:]+integrand[:-1])*.5*np.diff(z)))
gap=shape_gap(40001);refined=shape_gap(80001)
np.testing.assert_allclose(gap,refined,atol=1e-11)
np.testing.assert_allclose(gap,.0224274362,atol=1e-10)
print('moment-only Bayes epsilon gap:',gap,'grid change:',abs(gap-refined))
print('ordinary and mixed codes both resolve the example; no mixed-code superiority')

## Shared-checkpoint information versus finite accessibility
The actual matched method sends R to B1-aux and T(R) to M. This is a nested comparison, not a universal ordering between non-nested compressors.

In [ ]:
r=np.array([-1.,0.,1.]); target=r*r; message=r*r
x=np.column_stack([np.ones(3),r]); xm=np.column_stack([np.ones(3),message])
pred=x@np.linalg.lstsq(x,target,rcond=None)[0]
predm=xm@np.linalg.lstsq(xm,target,rcond=None)[0]
risk=float(np.mean((target-pred)**2));riskm=float(np.mean((target-predm)**2))
np.testing.assert_allclose(risk,2/9,atol=1e-12)
assert riskm<1e-25
lossy_gap=float(np.mean(r*r))
np.testing.assert_allclose(lossy_gap,2/3,atol=1e-12)
print('same Bayes information, affine risk R / T(R):',risk,riskm)
print('alternate target: irreversible Bayes gap:',lossy_gap)

## Batch normalization is a random ratio
Native t, noise, masks, reduction and the realized denominator are checked by native_forward, not replaced by a formula mirror.

In [ ]:
from itertools import product
cases=[(1.,0.),(9.,1.)]
ratios=[sum(w*d for w,d in batch)/sum(w for w,d in batch) for batch in product(cases,repeat=2)]
actual=float(np.mean(ratios));ratio_of_expectations=sum(w*d for w,d in cases)/sum(w for w,d in cases)
np.testing.assert_allclose(actual,.7)
np.testing.assert_allclose(ratio_of_expectations,.9)
print('expected batch ratio / ratio of expectations:',actual,ratio_of_expectations)

These finite checks retain the earlier Gaussian results. They do not execute HSE/reference extraction, prove target-domain calibration or show a learned LLapDiff advantage. Native component acceptance and genuine-export training are separate stages in paper/GOAL.md.

In [ ]:
print("THEORY_DEMO_PASS::12_posterior_precision_distortion")